In [1]:
# First up, we need two things before diving into the code. 
# We need to create a Python enviornment, conda or python built-in  install following Python libraries;
# requests minsearch openai jupyter python-dotenv
 

In [2]:
# Then we need to connect to OpenAI for our llm, for that we need an OpenAI API key 
# Which you can get from platform.openai.com. 

# Create an account if you don't have one, takes a minute
# Click on API Keys on the left sidebar and create one

# Then create a .env file which is where we will store it, simply assign it to OPENAI_API_KEY = XXXXXXX

# Then let's import OpenAI into our application
from openai import OpenAI
openai_client = OpenAI()

In [3]:
# And we use dotenv, library to load environment variables from your .env file where
# OpenAI API key, into the environment you're working in.
from dotenv import load_dotenv
load_dotenv() 

from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# That's all the set up.
# Next up we scrape the data from the Harvard CS50 course, small dataset. 
# I won't go into detail about scrapping the data
# in this video but I'll explain what it does; 

# It grabs the data from the Harvard website, puts it into json format and categorises them... e.g. Certificates, Books, 
# And it will save it as cs50_faq.json file (see left sidebar of VSCode)

# And whatever data you want to apply RAG to, you will need to first transform it into json structure if it's not already. 
# When you scrape any website, there are 4 forms that data can be in, Raw HTML, Plain text like PDFs, 
# Hidden APIs or Raw Json
# if you're lucky the website you're scraping will have raw json then you can use it as it is. 




# SCRIPT TO SCRAPE WEBSITE and put data into json format. 
# import json
# import os
# from bs4 import BeautifulSoup
# from openai import OpenAI
# from pydantic import BaseModel, Field
# import requests

# # Initialize OpenAI client (requires OPENAI_API_KEY env variable set)
# client = OpenAI()


# # 1. Define the target JSON structure
# class FAQItem(BaseModel):
#     id: str = Field(description="Sequential ID, e.g., cs50_faq_001")
#     category: str = Field(
#         description="Specific category (e.g., Grading, Regret Clause, AI Policy, Project, Resubmission)"
#     )
#     question: str = Field(
#         description="A distinct, single-topic operational question"
#     )
#     answer: str = Field(
#         description="Exact rule, score, percentage, or policy details without omitting facts"
#     )


# class FAQList(BaseModel):
#     items: list[FAQItem]


# def extract_max_cs50_json():
#     url = "https://cs50.harvard.edu/x/2026/syllabus/"
#     print(f"1. Fetching syllabus page from {url}...")

#     response = requests.get(url)
#     response.raise_for_status()

#     # 2. Clean HTML clutter
#     soup = BeautifulSoup(response.text, "html.parser")
#     for clutter in soup(["script", "style", "nav", "footer", "header", "aside"]):
#         clutter.extract()

#     clean_text = soup.get_text(separator="\n", strip=True)

#     print(
#         "2. Extracting maximum atomic FAQ items via OpenAI (no collapsing allowed)..."
#     )

#     # 3. Request maximum density extraction using strict system instructions
#     completion = client.beta.chat.completions.parse(
#         model="gpt-4o-mini",
#         messages=[
#             {
#                 "role": "system",
#                 "content": (
#                     "You are an exhaustive data extraction assistant. Parse the provided CS50 syllabus "
#                     "and extract EVERY SINGLE rule, policy, requirement, threshold, and restriction "
#                     "as its own separate, atomic FAQ item.\n\n"
#                     "CRITICAL EXTRACTION RULES:\n"
#                     "1. DO NOT summarize or combine rules into broad umbrella topics.\n"
#                     "2. Extract as many distinct items as possible (target 15 to 25 items).\n"
#                     "3. Separate distinct policies into individual items:\n"
#                     "   - Minimum score cutoff (70%) as one item.\n"
#                     "   - What happens if you score below 70% as a separate item.\n"
#                     "   - Resubmission mechanics as one item.\n"
#                     "   - Regret Clause deadline window (72 hours) as one item.\n"
#                     "   - Regret Clause consequences as a separate item.\n"
#                     "   - Allowed use of duck.cs50.ai as one item.\n"
#                     "   - Ban on third-party commercial AI (ChatGPT, Copilot, Claude) as a separate item.\n"
#                     "   - Unreasonable collaboration vs reasonable collaboration as separate items.\n"
#                     "   - Final project implementation rules as one item.\n"
#                     "   - Final project video presentation rules as a separate item.\n"
#                     "   - edX Certificate vs Harvard Division of Continuing Education certificate as separate items."
#                 ),
#             },
#             {"role": "user", "content": clean_text[:25000]},
#         ],
#         response_format=FAQList,
#     )

#     # 4. Save extracted items
#     parsed_items = completion.choices[0].message.parsed.model_dump()["items"]
#     output_filename = "cs50_faq.json"

#     with open(output_filename, "w", encoding="utf-8") as f:
#         json.dump(parsed_items, f, indent=2, ensure_ascii=False)

#     print(
#         f"3. Success! Extracted {len(parsed_items)} granular FAQ items to {output_filename}"
#     )
#     print(f"File location: {os.path.abspath(output_filename)}")


# if __name__ == "__main__":
#     extract_max_cs50_json()

In [5]:
# So nnow that we have json data saved down, let's pull it into this project
# And to assign all these json dictoinaries to a variable, we can read in the json file like this

import json

# 1. Load the local JSON file directly
with open("cs50_faq.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

# 2. Check how many json dictionaries were loaded
len(documents)

24

In [6]:
# Let's look at the content of the json dictionaries.
# Talk briefly about categorisation and that it's information relating to the category. 
documents

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to be eligible for a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate from edX.'},
 {'id': 'cs50_faq_003',
  'category': 'Resubmission',
  'question': 'Are there any opportunities to resubmit work after initial submission?',
  'answer': 'You are welcome to submit the course’s ten problem sets and final project for automated feedback; however, resubmission mechanics are not explicitly detailed in the syllabus.'},
 {'id': 'cs50_faq_004',
  'category': 'De

In [7]:
# Now that we have the data prepared, we move to the first of RAG: Retrieval


# The Retrieval step involves INDEXING and applying SEARCH to the data

# Many SEARCH libraries ... Elasticsearch Solr ... heavy duty search libraries ... docker. 
# Our case small dataset ... minsearch lightweight version of EalsticSearch
# Library created by Alexey Grigorev; I'll link the repo below. 

# We import Index class from the minsearch ... to index our data 
# and initialise it with two fields: text_field and keyword_fields

from minsearch import Index

index = Index(
    text_fields=["question", "answer"],
    keyword_fields=["category"]
)
# text_fields are the fields you PERFORM SEARCH, it's the data that the llm will use to answer your question.
# (Pull up the cs50_faq)
# In our case it is the 'question' field, and 'answer' fields. 
# So in the question and answer field we will find our answer about the pass mark I need to get a cert from Harvard
# for the CS50 course.

# You may be wondering why not category field, it's because that is a keyword_field; 
# and these are field where you need AN EXACT MATCH FOR. In our case it is Deadlines, Certificates, and few others
# reaosn you need an EXACT MATCH, is you'll typically database query like: 
# SELECT * FROM table WHERE course = 'Deadlines' ; so you need the exact string to match

# keyword_fields are used to NARROW the search down, 
# In our case we care about things related to the Certificates, so we narrow our search down to the
# Certificates category. 

# index.fit() come from the scikit-learn library and it fits the model to the data.
# In our case the json dictionaries. 
index.fit(documents)

question = 'What is the exact percentage score required on CS50 problem sets to get a certificate?' 

index.search(question, filter_dict={'category': 'Certificates'})



# index.search(question, filter_dict={'category':'Certificates'})  # returns all the Q & A's only from the Certificates section

# # And above we have index.fit() which come from the scikit-learn library and it fits the model to the data.
# # In our case the json dictionaries. 

# # you can also specify how many results you want to rerturn by passing the num_results
# search_results =index.search(question, filter_dict={'category':'Certificates'}, num_results=5) # if we just want 5 results
# search_results

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to be eligible for a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate from edX.'}]

In [8]:
# So put the above search into a function that we can call.

# One other thing word going over is the boost keyword; you can put more weight on certain text_fields;
# So if your question had the word CS500 in it, and two different json dictionaries had CS500, one in the question only and the other in the 
# answer only the json dict with it in the question will be matched twice as high as the one with it in the question only. 
# Reminder it is relative to 1. 
def search(question, category='Certificates'):
    boost_dict={'question':2.0}
    filter_dict={'category':category}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [9]:
# Let's see what we get ; as expected we had two Certificates categories in our 24 json dictionaries, 
# and both are in our search's results.

search_results = search(question)  
search_results

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to be eligible for a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate from edX.'}]

In [10]:
# We move onto the A in RAG; Augmentation.

# This is where we build the prompt that we will send to the LLM, and we will use the search results above 
# to augment the prompt.

# The PROMPT has two parts; the first part is the system prompt, and the second part is the user prompt.

In [11]:
# First part; the system prompt are the instructions you send to the llm, in our case...
INSTRUCTIONS = """
Your task is to answer questions from the Harvard CS50 course
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [12]:
# Second part is the user prompt, this is the part that can change as you ask different questions, 
# and the context which we get form the search results above will change depending on the question.
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [13]:
# Next up, we turn the python dictionaries that we got from our search into one long string, this makes it
# easier for the LLM to read and understand.
# I'll copy over a function that I wrote before, it takes the search_results which is the two json dicitonaries 
# and it turns them into one long string. 

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["category"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip() 

build_context(search_results)

'Certificates\nQ: What is the minimum score required to be eligible for a verified certificate from edX?\nA: You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.\n\nCertificates\nQ: What happens if I score below 70% on a problem set or final project?\nA: If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate from edX.'

In [14]:
# Then we feed this long string into our prompt template that we saw before, that we will send to the llm. 
# Say search_result we got from part one, build_context that turns it into a long string 
# and the prompt, which simply formats the long string using the USER_PROMPT_TEMPLATE. 
# And we remove any trailing or leading whitespace with strip. 
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)

print(prompt)

# So what we is this prompt variable which holds the search_result that was fed into this USER_PROMPT_TEMPLATE.
# Notice how it looks almost like the json dictionary, but it is in string format now, for LLM to understand 
# it easily.

# The prompt is the bridge between search and the LLM. 

Question:
What is the exact percentage score required on CS50 problem sets to get a certificate?

Context:
Certificates
Q: What is the minimum score required to be eligible for a verified certificate from edX?
A: You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.

Certificates
Q: What happens if I score below 70% on a problem set or final project?
A: If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate from edX.


In [15]:
"""
Then the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, 
hence generation.
"""

'\nThen the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, \nhence generation.\n'

In [16]:
# Towards the start I demonstrated this set up to call OpenAI's gtp 5.4 model. 
# We are using the response API ... used to be chat completion API, but that is legacy. 
# Mention it because Grok and Gemini for example still use chat completion... if you connect to those models

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [17]:
# Two things that are relevant before running the final rag
# is the cost for prompting the llm with the prompt we created.

# Can see the cost here:
# https://developers.openai.com/api/docs/models/gpt-5.4-mini
# Can see input and output price per million tokens

# in our case 
response.usage


ResponseUsage(input_tokens=140, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=63, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=203)

In [18]:
# Some basic calculations, saw it's 75p for 1m input tokens and 4.5 for 1m output tokens.

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

# a fraction of a penny... very cheap model. 

0.0003885

In [19]:
# And the second thing is when you have a chat with ChatGPT Claude or Gemini, you have message history 
# The answer ChatGPT gives you further along the conversation is based on the previous messages you sent to it.

# We won't build a multi-turn chat here. But may do so in another video. 

# For now, we just look at the set up; 
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [20]:
# We can now put this together into an updated llm function.
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text



In [21]:
# Finally, we go full-circle and look at the rag we spoke about right at the start.
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("What is the exact percentage score required on CS50 problem sets to get a certificate?")
print(answer)


# Can play around with it, read thorugh the Harvard page and ask it questions. 
# This is of course a simple example of using a single web page as the context. 


# Much of this content I got from llm-zoomcamp so if you want to check it out you can, I'll leave a link in 
# the description. 


# If you liked this video, you may enjoy my next one which will take this RAG pipeline to the next level 
# by building True Agentic RAG. 
# We'll cover query routing, self-correcting mechanisms (CRAG) so the models knows  when retrieved data 
# isn't good enough, handling multi-document reasoning, and using frameworks like Langraph to pull it 
# all together and evaluate its performance. See you there. 


You must receive **at least 70%** on each CS50 problem set to be eligible for a certificate.
